# CareLoop Clinics – Part 3: Machine Learning No-Show Prediction
**Tasks 3.1 – 3.5: Features, Baseline, Model Exploration, Honest CV Selection & Final Test Evaluation**

Objective: Predict appointment no-show risk on the **evening before** the appointment (after reminders are sent or skipped), using only pre-appointment features available at that time.

## Task 3.1 – Features, Split and Baseline (5 Marks)
Load `appointments_clean.csv` without sorting or shuffling, preserve `appointment_id` and its index, encode `reminder_sent` and one-hot encode `booking_channel` and `visit_type`, split 80/20 (`random_state=42`, no `stratify`), and compute the majority-class baseline.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def resolve_file(filename):
    search_dirs = [
        ".", "/content", "..",
        "data/processed", "/content/data/processed", "../data/processed",
        "CapstoneProjectMasai/data/processed",
    ]
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    for root_dir in (".", "/content"):
        if os.path.isdir(root_dir):
            for r, _, files in os.walk(root_dir):
                if filename in files:
                    return os.path.join(r, filename)
    raise FileNotFoundError(
        f"Could not find {filename} under {os.getcwd()} or /content. "
        f"Files in {os.getcwd()}: {os.listdir('.')}. "
        f"Please upload {filename} into Colab's /content/ folder."
    )

PROJECT_ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# STEP A - Load data without sorting or shuffling
df = pd.read_csv(resolve_file("appointments_clean.csv"))
print("Loaded shape:", df.shape)
print("Total no-shows:", int(df["no_show"].sum()), f"({df['no_show'].mean() * 100:.1f}%)")

# STEP B - Preserve appointment IDs aligned by index
appointment_ids = df["appointment_id"].copy()

# STEP C - Target
y = df["no_show"]

# STEP D - Valid pre-appointment features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type",
    ]
].copy()

X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# STEP E - One-hot encoding
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])

print("\nFeature names (11 expected):", list(X.columns))
print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())

# STEP F - Train/test split (no stratify, no shuffling before split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# STEP G - Report split information
test_no_shows = int((y_test == 1).sum())
test_attended = int((y_test == 0).sum())
print(f"\nTraining rows: {len(X_train)}")
print(f"Testing rows : {len(X_test)}")
print(f"Test no-shows: {test_no_shows}")
print(f"Test attended: {test_attended}")

# STEP H - Baseline accuracy (always predict 0 = Attended)
baseline_acc = accuracy_score(y_test, np.zeros(len(y_test)))
print(f"Baseline accuracy (always Attended): {baseline_acc:.4f}")

Loaded shape: (990, 16)
Total no-shows: 218 (22.0%)

Feature names (11 expected): ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Feature matrix shape: (990, 11)
Target distribution:
 no_show
0    772
1    218
Name: count, dtype: int64

Training rows: 792
Testing rows : 198
Test no-shows: 49
Test attended: 149
Baseline accuracy (always Attended): 0.7525


**Why accuracy alone is insufficient:** Because ~78% of appointments are attended (75.25% in the test split), a trivial model that predicts *Attended* for every booking achieves 75.25% accuracy while catching **0%** of actual no-shows (`recall = 0`). Metrics like recall, precision, F1-score, and ROC-AUC are essential to measure how well the model identifies missed appointments.

## Task 3.2 – Logistic Regression (6 Marks)
Fit `StandardScaler` on `X_train` only, train `LogisticRegression(max_iter=1000)`, evaluate on `X_test_s`, inspect coefficients in exact feature order, and compute both the scaled and unscaled (`0 -> 1`) reminder odds multipliers.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# STEP A - Feature scaling (fit ONLY on X_train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# STEP B - Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

# STEP C - Predictions
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:, 1]

# STEP D - Evaluation metrics
acc_lr = accuracy_score(y_test, y_pred_lr)
cm_lr = confusion_matrix(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print(f"Accuracy : {acc_lr:.3f}")
print(f"Precision: {prec_lr:.3f}")
print(f"Recall   : {rec_lr:.3f}")
print(f"F1-score : {f1_lr:.3f}")
print(f"ROC-AUC  : {auc_lr:.3f}")
print("\nConfusion Matrix (rows=actual [0=Attended, 1=No-Show], cols=predicted [0, 1]):")
print(pd.DataFrame(cm_lr, index=["Actual 0 (Attended)", "Actual 1 (No-Show)"], columns=["Pred 0 (Attended)", "Pred 1 (No-Show)"]))

# STEP E - Coefficients in exact column order of X
coef_df = pd.DataFrame({
    "Feature": list(X.columns),
    "Coefficient": lr_model.coef_[0],
})
coef_df["Interpretation"] = np.where(
    coef_df["Coefficient"] > 0.05,
    "Higher value associated with higher no-show odds",
    np.where(
        coef_df["Coefficient"] < -0.05,
        "Higher value associated with lower no-show odds",
        "Weak / near-zero association",
    ),
)
print("\nLogistic Regression Coefficients:")
print(coef_df.to_string(index=False))

# STEP F - Reminder odds ratio
reminder_idx = list(X.columns).index("reminder_sent")
reminder_coef = lr_model.coef_[0][reminder_idx]
reminder_std = scaler.scale_[reminder_idx]
scaled_odds_mult = np.exp(reminder_coef)
reminder_effect = np.exp(reminder_coef / reminder_std)

print(f"\nreminder_sent index: {reminder_idx}")
print(f"reminder_sent scaled coefficient: {reminder_coef:.4f}")
print(f"exp(scaled coefficient) [1-SD multiplier]: {scaled_odds_mult:.4f}")
print(f"Standard deviation of reminder_sent: {reminder_std:.4f}")
print(f"exp(coef / std) [No=0 to Yes=1 multiplier]: {reminder_effect:.4f} (~{(1 - reminder_effect) * 100:.0f}% lower estimated odds)")

Accuracy : 0.788
Precision: 0.630
Recall   : 0.347
F1-score : 0.447
ROC-AUC  : 0.789

Confusion Matrix (rows=actual [0=Attended, 1=No-Show], cols=predicted [0, 1]):
                     Pred 0 (Attended)  Pred 1 (No-Show)
Actual 0 (Attended)                139                10
Actual 1 (No-Show)                  32                17

Logistic Regression Coefficients:
                Feature  Coefficient                                   Interpretation
                    age    -0.253880  Higher value associated with lower no-show odds
            distance_km     0.253448 Higher value associated with higher no-show odds
                fee_inr     0.452761 Higher value associated with higher no-show odds
              lead_days     0.614471 Higher value associated with higher no-show odds
         prior_no_shows     0.649612 Higher value associated with higher no-show odds
          reminder_sent    -0.580669  Higher value associated with lower no-show odds
    booking_channel_App    

**Interpretation (Task 3.2):** `reminder_sent` has the most negative coefficient (`~ -0.58`), while `lead_days` (`~ +0.60`) and `prior_no_shows` (`~ +0.60`) are clearly positive. Because features were standardised, `exp(coef)` (`~0.56`) reflects a 1-SD change, whereas `exp(coef / std)` (`~0.31`) reflects switching `reminder_sent` from `0` (No) to `1` (Yes)—about 69% lower estimated odds of no-show in this fitted model, holding other features constant. This is an observational association, not proof of causation.

## Task 3.3 – Decision Tree and Overfitting (5 Marks)
Train a full-depth `DecisionTreeClassifier(random_state=42)` and a `DecisionTreeClassifier(max_depth=3, random_state=42)` to demonstrate overfitting.

In [3]:
from sklearn.tree import DecisionTreeClassifier

# Model 1: Full-depth tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
dt_full_train_acc = accuracy_score(y_train, dt_full.predict(X_train))
dt_full_test_acc = accuracy_score(y_test, dt_full.predict(X_test))

# Model 2: Depth-3 tree
dt_d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_d3.fit(X_train, y_train)
dt_d3_train_acc = accuracy_score(y_train, dt_d3.predict(X_train))
dt_d3_test_acc = accuracy_score(y_test, dt_d3.predict(X_test))
dt_d3_rec = recall_score(y_test, dt_d3.predict(X_test))
dt_d3_auc = roc_auc_score(y_test, dt_d3.predict_proba(X_test)[:, 1])

print(f"Full-depth tree depth: {dt_full.get_depth()}")
dt_comp = pd.DataFrame([
    {"Model": "Full-depth Decision Tree", "Training Accuracy": round(dt_full_train_acc, 4), "Test Accuracy": round(dt_full_test_acc, 4)},
    {"Model": "Depth-3 Decision Tree", "Training Accuracy": round(dt_d3_train_acc, 4), "Test Accuracy": round(dt_d3_test_acc, 4)},
])
print("\nDecision Tree Comparison:")
print(dt_comp.to_string(index=False))

Full-depth tree depth: 15

Decision Tree Comparison:
                   Model  Training Accuracy  Test Accuracy
Full-depth Decision Tree              1.000         0.6717
   Depth-3 Decision Tree              0.822         0.7778


**Overfitting Explanation (Task 3.3):** The full-depth decision tree grows deep enough (`depth ~15`) to memorise noise and individual training examples, achieving 100% training accuracy but dropping sharply on unseen test bookings. Limiting `max_depth=3` constrains complexity, trading some training accuracy for substantially better generalisation on the test set.

## Task 3.4 – Random Forest and Model Comparison (6 Marks)
Train `RandomForestClassifier(n_estimators=100, random_state=42)`, report test metrics and top 3 feature importances, and build a 3-model reporting table (without using it to pick the winner).

In [4]:
from sklearn.ensemble import RandomForestClassifier

# STEP A & B - Train and predict
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# STEP C - Evaluate
rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest Training Accuracy: {rf_train_acc:.4f}")
print(f"Random Forest Test Accuracy    : {rf_test_acc:.4f}")
print(f"Random Forest Recall           : {rf_rec:.4f}")
print(f"Random Forest ROC-AUC          : {rf_auc:.4f}")

# STEP D - Top 3 feature importances
fi_df = (
    pd.DataFrame({"Feature": list(X.columns), "Importance": rf.feature_importances_})
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)
print("\nTop 3 Features by Random Forest Importance:")
print(fi_df.head(3).to_string(index=False))

# STEP E - 3-Model Reporting Comparison Table
comp_34 = pd.DataFrame([
    {"Model": "Logistic Regression", "Accuracy": round(acc_lr, 4), "Recall": round(rec_lr, 4), "ROC-AUC": round(auc_lr, 4)},
    {"Model": "Depth-3 Decision Tree", "Accuracy": round(dt_d3_test_acc, 4), "Recall": round(dt_d3_rec, 4), "ROC-AUC": round(dt_d3_auc, 4)},
    {"Model": "Random Forest", "Accuracy": round(rf_test_acc, 4), "Recall": round(rf_rec, 4), "ROC-AUC": round(rf_auc, 4)},
])
print("\nReporting Comparison (Test Set - NOT for model selection):")
print(comp_34.to_string(index=False))

Random Forest Training Accuracy: 1.0000
Random Forest Test Accuracy    : 0.7475
Random Forest Recall           : 0.3061
Random Forest ROC-AUC          : 0.7057

Top 3 Features by Random Forest Importance:
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739

Reporting Comparison (Test Set - NOT for model selection):
                Model  Accuracy  Recall  ROC-AUC
  Logistic Regression    0.7879  0.3469   0.7889
Depth-3 Decision Tree    0.7778  0.2245   0.6997
        Random Forest    0.7475  0.3061   0.7057


## Task 3.5 – Tune, Choose Model and Threshold, Then Test Once (8 Marks)
Perform 5-fold `GridSearchCV` on `X_train, y_train` only, compare all 4 candidates via 5-fold training CV ROC-AUC (using a `Pipeline` for Logistic Regression), select the threshold (`0.50` vs `0.35`) via OOF training probabilities, fit the winner once on `X_train`, evaluate once on `X_test`, and export `outputs/test_risk_scores.csv`.

In [5]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# STEP A - Random Forest hyperparameter tuning on training data ONLY
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8],
}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_train, y_train)
print("STEP A - GridSearchCV best_params_:", grid.best_params_)
print(f"STEP A - GridSearchCV best_score_ (5-fold CV ROC-AUC): {grid.best_score_:.4f}")

# STEP B - Compare 4 candidates using 5-fold training CV ROC-AUC
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
dt_d3_cv = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_base_cv = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tuned_cv = grid.best_estimator_

cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_dt = cross_val_score(dt_d3_cv, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(rf_base_cv, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf_tuned = grid.best_score_

cv_table = pd.DataFrame([
    {"Model": "Logistic Regression Pipeline", "Training CV ROC-AUC": round(cv_lr, 4)},
    {"Model": "Depth-3 Decision Tree", "Training CV ROC-AUC": round(cv_dt, 4)},
    {"Model": "Random Forest", "Training CV ROC-AUC": round(cv_rf, 4)},
    {"Model": "Tuned Random Forest", "Training CV ROC-AUC": round(cv_rf_tuned, 4)},
])
print("\nSTEP B - Training Cross-Validation ROC-AUC Comparison:")
print(cv_table.to_string(index=False))

# Select winner strictly by highest training CV ROC-AUC
# (Tie-breaker: prefer simpler/more interpretable Logistic Regression Pipeline if rounded scores tie)
candidates = [
    ("Logistic Regression Pipeline", cv_lr, lr_pipe),
    ("Tuned Random Forest", cv_rf_tuned, rf_tuned_cv),
    ("Random Forest", cv_rf, rf_base_cv),
    ("Depth-3 Decision Tree", cv_dt, dt_d3_cv),
]
winner_name, winner_cv_auc, winner = max(candidates, key=lambda item: item[1])
print(f"\nSelected winner (highest training CV ROC-AUC): {winner_name} ({winner_cv_auc:.4f})")

# STEP C - Threshold selection using Out-Of-Fold (OOF) training probabilities
oof_probs = cross_val_predict(
    winner, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]

thresh_rows = []
for thr in [0.50, 0.35]:
    oof_pred = (oof_probs >= thr).astype(int)
    thresh_rows.append({
        "Threshold": thr,
        "Precision": round(precision_score(y_train, oof_pred), 4),
        "Recall": round(recall_score(y_train, oof_pred), 4),
        "F1": round(f1_score(y_train, oof_pred), 4),
        "Flagged": int(oof_pred.sum()),
    })

thresh_df = pd.DataFrame(thresh_rows)
print("\nSTEP C - OOF Threshold Comparison:")
print(thresh_df.to_string(index=False))

selected_threshold = float(thresh_df.loc[thresh_df["F1"].idxmax(), "Threshold"])
print(f"Selected threshold (highest OOF F1): {selected_threshold:.2f}")

# STEP D - Fit winner on complete training set and evaluate ONCE on X_test
winner.fit(X_train, y_train)
test_probabilities = winner.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= selected_threshold).astype(int)

final_prec = precision_score(y_test, test_predictions)
final_rec = recall_score(y_test, test_predictions)
final_flagged = int(test_predictions.sum())
final_auc = roc_auc_score(y_test, test_probabilities)

print("\nSTEP D - FINAL Test Set Evaluation (evaluated ONCE):")
print(f"  Precision          : {final_prec:.4f}")
print(f"  Recall             : {final_rec:.4f}")
print(f"  Flagged            : {final_flagged}")
print(f"  ROC-AUC (continuous): {final_auc:.4f}")

# STEP E - Export test_risk_scores.csv aligned by X_test.index
test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": np.round(test_probabilities, 6),
    "flag": test_predictions,
})

risk_scores_path = os.path.join(OUTPUTS_DIR, "test_risk_scores.csv")
test_risk_scores.to_csv(risk_scores_path, index=False)

print(f"\nSTEP E - Exported {risk_scores_path}")
print("  Shape:", test_risk_scores.shape)
print("  Missing values:", int(test_risk_scores.isnull().sum().sum()))
print("  Flagged count:", int(test_risk_scores["flag"].sum()))
print("  Selected threshold:", selected_threshold)
print("\nFirst 5 rows:")
print(test_risk_scores.head(5).to_string(index=False))

STEP A - GridSearchCV best_params_: {'max_depth': 3, 'n_estimators': 200}
STEP A - GridSearchCV best_score_ (5-fold CV ROC-AUC): 0.7673

STEP B - Training Cross-Validation ROC-AUC Comparison:
                       Model  Training CV ROC-AUC
Logistic Regression Pipeline               0.7737
       Depth-3 Decision Tree               0.6784
               Random Forest               0.7466
         Tuned Random Forest               0.7673

Selected winner (highest training CV ROC-AUC): Logistic Regression Pipeline (0.7737)

STEP C - OOF Threshold Comparison:
 Threshold  Precision  Recall     F1  Flagged
      0.50     0.5753  0.2485 0.3471       73
      0.35     0.5130  0.4675 0.4892      154
Selected threshold (highest OOF F1): 0.35

STEP D - FINAL Test Set Evaluation (evaluated ONCE):
  Precision          : 0.5714
  Recall             : 0.5714
  Flagged            : 49
  ROC-AUC (continuous): 0.7889

STEP E - Exported .\outputs\test_risk_scores.csv
  Shape: (198, 4)
  Missing values:

**STEP F – Real-World Threshold Selection:**
A clinic would select the operational threshold based on how many follow-up calls the front desk can make in an evening, balanced against the cost of an empty doctor slot.

---
## Limitations (Documented for Part 4)
1. **Pre-split cleaning leakage:** The medians, quartiles, and outlier fences in Task 1.3 were calculated using all 990 rows before splitting; therefore, test rows had a small indirect influence on cleaning. In a real deployment, these statistics should be calculated using training data only.
2. **Repeat patients across splits:** The same `patient_id` can appear in multiple bookings, so a random split can place the same patient in both training and testing datasets.
3. **Random vs. temporal split:** The split is random, whereas a real deployment would train on historical data and predict future appointments; a temporal validation strategy would be more representative of production performance.